# v16 · de Bruijn traversal + Cayley–Dickson fold to a fixed-width address
Branch `data-storage-no-location` of `FourthAgePapers` (2026-08-30; local, unpushed): `DataStorageNoLocation/engine/bench_tower.py`. The files are read out of the branch with `git show`, unmodified, and run as a script.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import subprocess, tempfile, pathlib, re
repo = os.path.join(PROJ, "FourthAgePapers")
def show(path): return subprocess.run(["git", "-C", repo, "show", "data-storage-no-location:" + path], capture_output=True, text=True, check=True).stdout
d = pathlib.Path(tempfile.mkdtemp()); (d / "engine").mkdir(); (d / "bench").mkdir()      # the script writes ../bench/rung0_address.txt
(d / "engine" / "bench_tower.py").write_text(show("DataStorageNoLocation/engine/bench_tower.py"))
(d / "bench" / "cover_letter.txt").write_text(show("DataStorageNoLocation/bench/cover_letter.txt"))
print("branch files:", subprocess.run(["git", "-C", repo, "ls-tree", "-r", "--name-only", "data-storage-no-location", "DataStorageNoLocation"], capture_output=True, text=True).stdout.split())
print("document:", len((d / "bench" / "cover_letter.txt").read_bytes()), "bytes")

branch files: ['DataStorageNoLocation/README.md', 'DataStorageNoLocation/bench/cover_letter.txt', 'DataStorageNoLocation/bench/results.txt', 'DataStorageNoLocation/bench/rung0_address.txt', 'DataStorageNoLocation/construction.json', 'DataStorageNoLocation/engine/bench_tower.py']
document: 3360 bytes


In [3]:
t0 = time.perf_counter()
out = subprocess.run([sys.executable, "bench_tower.py", "../bench/cover_letter.txt"], cwd=d / "engine", capture_output=True, text=True, timeout=900)
print("exit", out.returncode, "| %.0f s" % (time.perf_counter() - t0))
assert out.returncode == 0, out.stderr[-300:]
lines = out.stdout.splitlines()
i0 = next(i for i, l in enumerate(lines) if "time per data size" in l)
print("\n".join(lines[i0:i0 + 13]))
start = next(i for i, l in enumerate(lines) if l.startswith("rung  alg"))
print("\n".join(lines[start:start + 8]))

exit 0 | 95 s
── time per data size ──
  reference : ThinkPad X1 Carbon 6th gen · Intel Core i7-8650U (4C/8T, Kaby Lake R, 1.9 GHz base / 4.2 GHz turbo, 8 MB L3) · 16 GB LPDDR3-2133 · NVMe SSD
  this run  : Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz · 8 logical CPUs · Linux 6.8.0-139-lowlatency

     bytes   Horner_bits   t_Horner(ms)   t_ℂ(ms)   t_ℍ(ms)   t_𝕆(ms)   t_𝕊(ms)
  ────────   ───────────   ────────────   ───────   ───────   ───────   ───────
       512         4,095          0.170     4.765    24.368   110.066   450.826
      1024         8,191          0.557     8.840    52.752   253.344   938.255
      2048        16,383          2.147    29.288   109.118   438.194   1765.371
      4096        32,767          9.073    35.062   202.955   858.722   3564.843
      8192        65,535         25.444    68.792   370.969   1700.259   7104.867
     16384       131,071         97.274   140.636   752.892   3424.563   16679.010
     32768       262,143        368.956   344.204   1492.6

Rung 0 (ℝ) is the raw bijection: its address grows with the document. Rungs ℂ → 𝕊 fold the same document into a **fixed 64-bit** address by the Cayley–Dickson product of the de Bruijn window sequence; time grows with the rung. The fold is not lossless for arbitrary documents (2⁶⁴ outputs, unbounded inputs); it is an *address* of the document that identifies what was stored. The 'reversible' column of the script refers to running the fold's own inverse on the stored windows.